# Solving Durable model with taste shocks in infinite horizon

Compare steady-state aggregates and sequence-space Jacobians w.r.t. `Rfree` and `labor` across
EV taste-shock magnitudes matching `Figures_Durable.ipynb` panel (a):

- $\sigma_\zeta \in \{0.1, 0.15, 0.2\}$
- Symmetric EV Type I taste shocks (`kappa=0`). The Zorzi–Beraja utility cost of adjusting is off.
- Search and price shut off (`SearchCost=0`, `PriceShkStd=0`)

Saves results to `Output/DurableModel_Taste_Inf_SS.pkl`,
`Output/DurableModel_Taste_Inf_Jacobians_Rfree.pkl`, and
`Output/DurableModel_Taste_Inf_Jacobians_labor.pkl`.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import copy
import time
import datetime
import pickle
from pathlib import Path

import pandas as pd
from IPython.display import display

start_time = time.time()

from DurableModel_Fast import DurableConsumer

FIG_DIR = Path("../Figures/DurableModel")
FIG_DIR.mkdir(parents=True, exist_ok=True)

OUT_DIR = Path("../Output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Ordered sweeps, low to high. logs/20261015_Figures.txt.
SWEEP_COLORS = ("#000000", "#0072B2", "#D55E00", "#009E73")
SWEEP_LW = 1.6
N_SHOW_DEFAULT = (0.5, 1.0, 1.5, 2.0)
_KEEP_TINT = (240 / 255, 240 / 255, 240 / 255)
_ADJ_TINT = (214 / 255, 230 / 255, 245 / 255)


def _legend_under_title(ax, handles, labels):
    """Legend row under the axes title, above the plot, frame off."""
    if not handles:
        return
    ax.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.0),
        ncol=len(labels),
        frameon=False,
        borderaxespad=0.25,
    )
    title = ax.get_title()
    if title:
        ax.set_title(title, pad=20)


def _legend_under_suptitle(fig, handles, labels):
    """One legend for the figure, under the suptitle."""
    if not handles:
        return
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.01),
        ncol=len(labels),
        frameon=False,
        borderaxespad=0.0,
    )



def save_pdf(fig, path):
    """Write the figure, then replace path.

    Windows error 22 on savefig means the destination could not be opened
    for writing. A PDF viewer that still has the file open does that.
    """
    path = Path(path).resolve()
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name("." + path.stem + ".writing.pdf")
    fig.savefig(tmp, dpi=150, bbox_inches="tight")
    try:
        tmp.replace(path)
    except OSError as err:
        raise OSError(
            "Could not replace %s. Close that PDF if a viewer has it open. "
            "The new figure was written to %s." % (path, tmp)
        ) from err


SS_KEYS = ["C", "D", "DEX", "TEX", "A", "ADJ"]
JAC_KEYS = ["C", "DEX", "TEX", "D", "A", "ADJ"]

JAC_NAMES = {
    "C": "nondurable consumption",
    "DEX": "durable expenditure",
    "TEX": "total expenditure",
    "D": "durable stock",
    "A": "assets",
    "ADJ": "adjustment probability",
}


In [2]:
### Baseline parameters for all models

durable_defaults = dict(
    CRRA=2.0,
    DiscFac=0.975,
    LivPrb=1.0,
    alpha=0.75,
    d_ubar=1e-4,
    dDepr=0.05,
    adjC=0.10,
    MaintCost=0.0,
    OpCost=0.0,
    Rfree=1.04**(0.25),
    RSpread=0.0,
    PermGroFac=1.0,
    LTV=0.0,
    BoroCnstArt=0.0,
    # Income
    PermShkStd=0.0,
    PermShkCount=1,
    TranShkStd=0.2,
    TranShkCount=7,
    UnempPrb=0.0,
    IncUnemp=0.0,
    IncUnemp_Std=0.1,
    truncation=2,
    taxrate=0.0,
    labor=1.0,
    wage=1.0,
    # Channels (EV taste sigma_eps vs search are mutually exclusive).
    # kappa=0 is the symmetric logit. A positive kappa is the Zorzi–Beraja
    # utility cost of adjusting and is not used in this notebook.
    sigma_eps=0.0,
    kappa=0.0,
    SearchCost=0.1,
    ScaleBool=False,
    e_grid_count=101,
    PriceShkCount=5,
    PriceShkStd=0.1,
    # EoP method
    use_expected=True,
    use_TransitionMatrix=False,
    # Grids
    nNrmMax=10, #5.0,
    nNrmCount=200,
    mNrmMax=16.0, #8.0,
    mNrmCount=200,
    xNrmMax=26, #13.0,
    xNrmCount=100,
    aNrmMax=16.0, #8.0,
    aNrmCount=200,
    NestFac=1,
    grid_type="exp_mult",
    TranMatrix_mCount=1010,
    TranMatrix_bCount=1000,
    N_scan=800,
    cycles=0,
    T_cycle=1,       # number of periods before terminal when cycles >= 1
    tol=1e-8,        # infinite-horizon convergence tolerance (matches background)
    max_cycles=500,  # iteration cap for cycles=0 (matches background max_cycles)
    min_cycles=200,  # minimum IH iterations before tol can stop
    distance_metric="mse",
    verbose=False,
)


In [3]:
# Symmetric EV taste shock only (kappa=0): no search, no price, no utility cost
# of adjusting. Matches Figures_Durable panel (a).
base_taste = copy.deepcopy(durable_defaults)
base_taste["sigma_eps"] = 0.0
base_taste["kappa"] = 0.0
base_taste["SearchCost"] = 0.0
base_taste["PriceShkStd"] = 0.0
base_taste["PriceShkCount"] = 1
base_taste["TranMatrixmCount"] = 250
base_taste["TranMatrixmMax"] = durable_defaults["mNrmMax"]
base_taste["TranMatrixnMax"] = durable_defaults["nNrmMax"]
base_taste["diagnostics"] = False

# Same grid as Figures_Durable.ipynb smoothing shifts
eps_vals = [0.1, 0.15, 0.2]
param_labels = [rf"$\sigma_{{\zeta}} = {sx}$" for sx in eps_vals]

T = 50


In [4]:
results = []
print(f"Taste specification: symmetric EV (kappa={base_taste['kappa']})")
for sx, lab in zip(eps_vals, param_labels):
    print(f"Solving / SS / Jacobian: sigma_eps={sx}...")
    params = copy.deepcopy(base_taste)
    params["sigma_eps"] = sx
    assert params["kappa"] == 0.0
    model = DurableConsumer(**params)
    assert model.kappa == 0.0 and model.sigma_eps == sx
    model.compute_steady_state()
    print(
        f"  converged in {model.completed_cycles} cycles, "
        f"{model.distance_metric} distance = {model.distance:.3e}"
    )    # Kinks already on solution[0] from solve; fill Judd Euler residuals.
    model.compute_euler_errors()
    sol0 = model.solution[0]
    kink_euler = {
        "Specification": lab,
        "Decr. seg.": sol0.keep_decreasing_segments,
        "Keep share": sol0.keep_share_decreasing,
        "FUES rem.": sol0.adj_kinks_removed,
        "Adj share": sol0.adj_share_removed,
        "EE_c keep mean": sol0.ee_c_keep_mean,
        "EE_c keep p95": sol0.ee_c_keep_p95,
        "EE_c adj mean": sol0.ee_c_adj_mean,
        "EE_c adj p95": sol0.ee_c_adj_p95,
        "EE_d adj mean": sol0.ee_d_adj_mean,
        "EE_d adj p95": sol0.ee_d_adj_p95,
        "n keep": getattr(sol0, "ee_c_keep_n", None),
        "n adj": getattr(sol0, "ee_d_adj_n", None),
    }
    J_C_Rfree, J_DEX_Rfree, J_TEX_Rfree, J_D_Rfree, J_A_Rfree, J_ADJ_Rfree = model.calc_jacobian("Rfree", T)
    J_C_labor, J_DEX_labor, J_TEX_labor, J_D_labor, J_A_labor, J_ADJ_labor = model.calc_jacobian("labor", T)
    results.append(
        dict(
            channel="taste",
            param_name="sigma_eps",
            param_col=r"$\sigma_\zeta$",
            param=sx,
            label=lab,
            kink_euler=kink_euler,
            ss=dict(
                C=float(model.C_ss),
                D=float(model.D_ss),
                DEX=float(model.DEX_ss),
                TEX=float(model.TEX_ss),
                A=float(model.A_ss),
                ADJ=float(model.ADJ_ss),
            ),
            jacobians_Rfree=dict(C=J_C_Rfree, DEX=J_DEX_Rfree, TEX=J_TEX_Rfree, D=J_D_Rfree, A=J_A_Rfree, ADJ=J_ADJ_Rfree),
            jacobians_labor=dict(C=J_C_labor, DEX=J_DEX_labor, TEX=J_TEX_labor, D=J_D_labor, A=J_A_labor, ADJ=J_ADJ_labor),
        )
    )
print("Finished all taste-shock variants.")


Taste specification: symmetric EV (kappa=0.0)
Solving / SS / Jacobian: sigma_eps=0.1...


C:\Users\amonninger\AppData\Local\miniconda3\envs\DCEGM_Search_env\lib\site-packages\HARK\rewards.py:40: RuntimeWarning: divide by zero encountered in reciprocal
  return c ** (1.0 - rho) / (1.0 - rho)
C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel_Fast.py:1233: RuntimeWarning: divide by zero encountered in power
  * C ** (self.alpha * (1.0 - self.CRRA) - 1.0)
C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel_Fast.py:1240: RuntimeWarning: divide by zero encountered in power
  * (C ** (self.alpha * (1.0 - self.CRRA)))
C:\Users\amonninger\AppData\Local\miniconda3\envs\DCEGM_Search_env\lib\site-packages\HARK\rewards.py:184: RuntimeWarning: divide by zero encountered in power
  return uP ** (-1.0 / rho)


C:\Users\amonninger\AppData\Local\miniconda3\envs\DCEGM_Search_env\lib\site-packages\HARK\rewards.py:40: RuntimeWarning: divide by zero encountered in reciprocal
  return c ** (1.0 - rho) / (1.0 - rho)
C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel_Fast.py:1240: RuntimeWarning: divide by zero encountered in power
  * (C ** (self.alpha * (1.0 - self.CRRA)))
C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel_Fast.py:1233: RuntimeWarning: divide by zero encountered in power
  * C ** (self.alpha * (1.0 - self.CRRA) - 1.0)


C:\Users\amonninger\AppData\Local\miniconda3\envs\DCEGM_Search_env\lib\site-packages\HARK\rewards.py:40: RuntimeWarning: divide by zero encountered in reciprocal
  return c ** (1.0 - rho) / (1.0 - rho)
C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel_Fast.py:1233: RuntimeWarning: divide by zero encountered in power
  * C ** (self.alpha * (1.0 - self.CRRA) - 1.0)
C:\Users\amonninger\AppData\Local\miniconda3\envs\DCEGM_Search_env\lib\site-packages\HARK\rewards.py:184: RuntimeWarning: divide by zero encountered in power
  return uP ** (-1.0 / rho)
C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel_Fast.py:1240: RuntimeWarning: divide by zero encountered in power
  * (C ** (self.alpha * (1.0 - self.CRRA)))


  converged in 200 cycles, mse distance = 1.072e-15


...

Solving / SS / Jacobian: sigma_eps=0.15...


  converged in 200 cycles, mse distance = 1.018e-16


Solving / SS / Jacobian: sigma_eps=0.2...


  converged in 200 cycles, mse distance = 9.691e-17


Finished all taste-shock variants.


In [5]:
# Steady-state comparison table
ss_rows = []
for r in results:
    row = {r["param_col"]: r["param"]}
    row.update(r["ss"])
    ss_rows.append(row)

ss_table = pd.DataFrame(ss_rows).set_index(results[0]["param_col"])
ss_table = ss_table[SS_KEYS]
print("Steady-state aggregates by EV taste scale")
display(ss_table.style.format("{:.6f}"))


Steady-state aggregates by EV taste scale


,C,D,DEX,TEX,A,ADJ
$\sigma_\zeta$,,,,,,
0.100000,0.795323,3.563025,0.178151,0.973474,0.935043,0.121865
0.150000,0.789397,3.266313,0.163316,0.952713,0.708470,0.185730
0.200000,0.785099,3.034149,0.151707,0.936806,0.604410,0.246915


In [6]:
# Save steady states and Jacobians as separate pickles
ss_payload = {
    "channel": results[0]["channel"],
    "param_name": results[0]["param_name"],
    "param_col": results[0]["param_col"],
    "param_values": [r["param"] for r in results],
    "labels": [r["label"] for r in results],
    "steady_states": {r["param"]: r["ss"] for r in results},
    "kink_euler": {r["param"]: r["kink_euler"] for r in results},
    "kink_euler_t": 0,
    "ss_table": ss_table,
}

jac_meta = dict(
    channel=results[0]["channel"],
    param_name=results[0]["param_name"],
    param_col=results[0]["param_col"],
    param_values=[r["param"] for r in results],
    labels=[r["label"] for r in results],
    T=T,
    shock_s_plotted=0,
)

jac_payload_Rfree = {
    **jac_meta,
    "shk_param": "Rfree",
    "jacobians": {r["param"]: r["jacobians_Rfree"] for r in results},
}
jac_payload_labor = {
    **jac_meta,
    "shk_param": "labor",
    "jacobians": {r["param"]: r["jacobians_labor"] for r in results},
}

ss_path = OUT_DIR / "DurableModel_Taste_Inf_SS.pkl"
jac_path_Rfree = OUT_DIR / "DurableModel_Taste_Inf_Jacobians_Rfree.pkl"
jac_path_labor = OUT_DIR / "DurableModel_Taste_Inf_Jacobians_labor.pkl"
with open(ss_path, "wb") as f:
    pickle.dump(ss_payload, f)
with open(jac_path_Rfree, "wb") as f:
    pickle.dump(jac_payload_Rfree, f)
with open(jac_path_labor, "wb") as f:
    pickle.dump(jac_payload_labor, f)

elapsed = time.time() - start_time
print(f"Saved {ss_path.resolve()}")
print(f"Saved {jac_path_Rfree.resolve()}")
print(f"Saved {jac_path_labor.resolve()}")
print("Done in", str(datetime.timedelta(seconds=elapsed)))


Saved C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Output\DurableModel_Taste_Inf_SS.pkl
Saved C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Output\DurableModel_Taste_Inf_Jacobians_Rfree.pkl
Saved C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Output\DurableModel_Taste_Inf_Jacobians_labor.pkl
Done in 0:01:20.874416


In [7]:
# Overlay Jacobians across parameter values (IRF to period-0 shock)
# Linestyles follow Figures_Durable.ipynb smoothing overlays.
shock_s = 0  # column of Jacobian = date of shock

# --- Rfree ---
SUPTITLE_R = r'Jacobians w.r.t. Rfree (shock at $s=0$), $\sigma_{\zeta} \in \{0.1, 0.15, 0.2\}$'
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, name in zip(axes.flat, JAC_KEYS):
    for r, color in zip(results, SWEEP_COLORS):
        J = r["jacobians_Rfree"][name]
        ax.plot(
            J[:, shock_s], color=color, linestyle="-",
            linewidth=SWEEP_LW, label=r["label"],
        )
    ax.set_title(JAC_NAMES[name][:1].upper() + JAC_NAMES[name][1:])
    ax.set_xlabel("period t")
    ax.set_ylabel("response of " + JAC_NAMES[name])
fig.suptitle(SUPTITLE_R, y=1.14)
handles, labs = axes.flat[0].get_legend_handles_labels()
_legend_under_suptitle(fig, handles, labs)
fig.tight_layout()
save_pdf(fig, FIG_DIR / "Taste_Inf_Jacobians_Rfree.pdf")
plt.show()

# --- labor ---
SUPTITLE_L = r'Jacobians w.r.t. labor (shock at $s=0$), $\sigma_{\zeta} \in \{0.1, 0.15, 0.2\}$'
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, name in zip(axes.flat, JAC_KEYS):
    for r, color in zip(results, SWEEP_COLORS):
        J = r["jacobians_labor"][name]
        ax.plot(
            J[:, shock_s], color=color, linestyle="-",
            linewidth=SWEEP_LW, label=r["label"],
        )
    ax.axhline(0.0, color="0.45", lw=0.6, zorder=0)
    ax.set_title(JAC_NAMES[name][:1].upper() + JAC_NAMES[name][1:])
    ax.set_xlabel("period t")
    ax.set_ylabel("response of " + JAC_NAMES[name])
fig.suptitle(SUPTITLE_L, y=1.14)
handles, labs = axes.flat[0].get_legend_handles_labels()
_legend_under_suptitle(fig, handles, labs)
fig.tight_layout()
save_pdf(fig, FIG_DIR / "Taste_Inf_Jacobians_labor.pdf")
plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_28596\2145199382.py:23: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_28596\2145199382.py:44: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [8]:
# Overlay Jacobians across parameter values (IRF to period-0 shock)
# Linestyles follow Figures_Durable.ipynb smoothing overlays.
shock_s = 20  # column of Jacobian = date of shock

# --- Rfree ---
SUPTITLE_R = r'Jacobians w.r.t. Rfree (shock at $s=0$), $\sigma_{\zeta} \in \{0.1, 0.15, 0.2\}$'
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, name in zip(axes.flat, JAC_KEYS):
    for r, color in zip(results, SWEEP_COLORS):
        J = r["jacobians_Rfree"][name]
        ax.plot(
            J[:, shock_s], color=color, linestyle="-",
            linewidth=SWEEP_LW, label=r["label"],
        )
    ax.set_title(JAC_NAMES[name][:1].upper() + JAC_NAMES[name][1:])
    ax.set_xlabel("period t")
    ax.set_ylabel("response of " + JAC_NAMES[name])
fig.suptitle(SUPTITLE_R, y=1.14)
handles, labs = axes.flat[0].get_legend_handles_labels()
_legend_under_suptitle(fig, handles, labs)
fig.tight_layout()
save_pdf(fig, FIG_DIR / "Taste_Inf_Jacobians_Rfree_t20.pdf")
plt.show()

# --- labor ---
SUPTITLE_L = r'Jacobians w.r.t. labor (shock at $s=0$), $\sigma_{\zeta} \in \{0.1, 0.15, 0.2\}$'
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, name in zip(axes.flat, JAC_KEYS):
    for r, color in zip(results, SWEEP_COLORS):
        J = r["jacobians_labor"][name]
        ax.plot(
            J[:, shock_s], color=color, linestyle="-",
            linewidth=SWEEP_LW, label=r["label"],
        )
    ax.axhline(0.0, color="0.45", lw=0.6, zorder=0)
    ax.set_title(JAC_NAMES[name][:1].upper() + JAC_NAMES[name][1:])
    ax.set_xlabel("period t")
    ax.set_ylabel("response of " + JAC_NAMES[name])
fig.suptitle(SUPTITLE_L, y=1.14)
handles, labs = axes.flat[0].get_legend_handles_labels()
_legend_under_suptitle(fig, handles, labs)
fig.tight_layout()
save_pdf(fig, FIG_DIR / "Taste_Inf_Jacobians_labor_t20.pdf")
plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_28596\3006612533.py:23: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_28596\3006612533.py:44: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [9]:
model.solution[0].xAdj_pre_list

[array([ 4.26105285,  4.41259226,  4.71133273,  5.19050032,  5.89284514,
         6.83887597,  8.01912798,  9.41587317, 11.01246893, 12.79668537,
        14.75646731, 16.88335578, 19.1708319 , 21.60680897])]

In [10]:
plt.scatter(model.solution[0].xAdj_post_list, model.solution[0].cAdj_post_list)

In [11]:
model.solution[0].xAdj_post_list

[array([ 4.26105285,  4.41259226,  4.71133273,  5.19050032,  5.89284514,
         6.83887597,  8.01912798,  9.41587317, 11.01246893, 12.79668537,
        14.75646731, 16.88335578, 19.1708319 , 21.60680897])]